# Correlation analysis - Single Video

One row per **detected face**, for **every** frame. Audio is left-joined (NaN where a frame has no clean segment) and flagged with `has_audio`, so visual-only analyses use everything and audio analyses just filter.

In [ ]:
#imports
import pandas as pd
import os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

## Select Candidates

In [ ]:
# Select candidates 

person1 = 'Martins'
person2 = 'Gouveia_Melo'

## Pipeline alternative — optional shortcut using `pipelines.py`

Everything from the cell marked **▼ START** below down to the cell marked **▲ END** (the current cells 6–16, about 230 lines) builds the dataset by hand: it loads the pkl files, extracts features frame by frame, runs the GMM visual classifier, joins audio speaker labels, and aggregates everything to segment level.

Dinis extracted all of that logic into shared functions in `multivideo_analysis/pipelines.py`. If you want to try the centralised version, comment out all cells from **▼ START** to **▲ END** and run the commented code cell just below this text instead — it calls the same logic and produces the same `seg`, `emotion_cols`, and `pose_feature_cols` variables that Sections B, C, and D expect.

**Why Section A is a problem with the pipeline:**
Section A uses `ident`, which comes from `df_all` — a frame-level table (one row per detected face per frame) that also carries audio features and speaker identity. The pipeline function `build_seg_all()` builds that table internally but only *returns* `seg`, the segment-level summary. It throws away the frame-level rows on purpose: when running across all 28 debates keeping every frame in memory would be expensive. For a single-debate analysis like this one that is the wrong tradeoff, so for now Section A still needs the manual cells below. Everything from Section B onwards works perfectly with the pipeline.

In [ ]:
## ── PIPELINE ALTERNATIVE ─────────────────────────────────────────────────────
## Comment out cells 6–16 (from ▼ START to ▲ END), then uncomment this block.
## Requires multivideo_analysis/pipelines.py (run Jupyter from the project root).
##
# import sys, os
# sys.path.insert(0, '../multivideo_analysis')   # one level up from labeling_visual/
# from pipelines import (build_labeled_audio, label_debate_visual,
#                        build_seg_all, NON_REDUNDANT)
#
# # Build the exact video filename from person1 / person2 (defined in cell 4 above)
# video_name = next(
#     f.replace('_audio.pkl', '')
#     for f in os.listdir('Project_Features')
#     if f.endswith('_audio.pkl') and person1 in f and person2 in f
# )
#
# data_audio = build_labeled_audio()
# # ^ loads all 28 debates + runs k=3 speaker embedding clustering
#
# labels = label_debate_visual(video_name, data_audio)
# # ^ visual GMM classifier (landmark ratios -> candidate_left/right/moderator)
# #   then resolves to real names using audio speaker labels
#
# seg = build_seg_all([video_name], data_audio, labels)
# # ^ frame->audio join, identity merge, speaking_face flag, segment aggregation
# #   pass a list of more debate names here for multi-video analysis
#
# # replicate the two column-list variables defined in the manual cells
# emotion_cols      = [c for c in seg.columns if c.startswith('prob_')]
# pose_feature_cols = ['shoulder_slope', 'body_openness', 'head_tilt',
#                      'wrist_height', 'torso_height']
#
# # NOTE: df_all is not available — Section A (ident variable) will not run.
# #       Sections B, C, D work unchanged.

## Load visual + audio for this debate

> **▼ START — manual pipeline** (skip to **▲ END** after the segment-aggregation cell if using the pipeline alternative above)


In [ ]:
# ── Visual ────────────────────────────────────────────────────────────────────
pklfiles_visual = []
for file in os.listdir('Project_Features'):
    if file.endswith('visual.pkl') and person1 in file and person2 in file:
        pklfiles_visual.append(file)

visual_dfs = []
for video in pklfiles_visual:
    print(f"Loading visual: {video}")
    df = pd.read_pickle(os.path.join('Project_Features', video))
    visual_dfs.append(df)

df_visual = pd.concat(visual_dfs, ignore_index=True)
df_visual['video'] = df_visual['Frame'].str.extract(r'Frames/([^/]+)/')
df_visual['frame_number'] = df_visual['Frame'].str.extract(r'frame_(\d+)\.jpg').astype(int)
df_visual = df_visual.sort_values(by=['video', 'frame_number']).reset_index(drop=True)

# ── Audio ─────────────────────────────────────────────────────────────────────
pklfiles_audio = []
for file in os.listdir('Project_Features'):
    if file.endswith('audio.pkl') and person1 in file and person2 in file:
        pklfiles_audio.append(file)

audio_dfs = []
for video in pklfiles_audio:
    print(f"Loading audio: {video}")
    df = pd.read_pickle(os.path.join('Project_Features', video))
    df['video'] = video.replace('_audio.pkl', '')
    audio_dfs.append(df)

df_audio = pd.concat(audio_dfs, ignore_index=True)
df_audio = df_audio.sort_values(by=['video', 'time stamp']).reset_index(drop=True)

def parse_embedding(x):
    if isinstance(x, np.ndarray):
        return x
    return np.fromstring(str(x).strip('[]'), sep=' ')

df_audio['speak_embeddings'] = df_audio['speak_embeddings'].apply(parse_embedding)

## Per-frame audio — keep ALL frames
For each frame second, attach the single audio segment active then. Frames in a gap, or with overlapping segments, keep `has_audio=False` and NaN audio features (we don't drop them).

In [ ]:
NON_REDUNDANT = ['meanF0Hz','stdevF0Hz','HNR','localJitter','localShimmer',
                 'speechrate','npause','f1_mean','f2_mean','fdisp']

df_audio['time_end'] = df_audio['time stamp'] + df_audio['duration']

rows = []
for t in sorted(df_visual['frame_number'].unique()):
    active = df_audio[(df_audio['time stamp'] <= t) & (df_audio['time_end'] > t)]
    if len(active) == 1:                      # exactly one speaker this second
        seg = active.iloc[0]
        r = {'frame_number': int(t), 'has_audio': True, 'segment_id': int(active.index[0])}
        r.update({c: seg[c] for c in NON_REDUNDANT})
    else:                                     # gap (0) or overlap (>1) -> no clean audio
        r = {'frame_number': int(t), 'has_audio': False, 'segment_id': -1}
        r.update({c: np.nan for c in NON_REDUNDANT})
    rows.append(r)

audio_by_frame = pd.DataFrame(rows)
print('frames total      :', len(audio_by_frame))
print('frames with audio :', int(audio_by_frame['has_audio'].sum()))
print('distinct segments :', audio_by_frame.loc[audio_by_frame.has_audio, "segment_id"].nunique())

## Per-face visual table (faces matched to bodies + pose features)

In [ ]:
master_data = []
for index, row in df_visual.iterrows():
    frame_id = row['Frame']
    faces = row['Fer']
    poses = row['Poses'] # This contains the body bbox and the 17 keypoints
    frame_number = row['frame_number']
    
    if isinstance(faces, list) and len(faces) > 0 and len(faces)<=3:
        valid_faces = [f for f in faces if isinstance(f, dict) and 'bbox' in f]
        people_count = len(valid_faces)
        
        for i, face in enumerate(valid_faces):
            # 1. Face Coordinates
            f_box = face['bbox'] # [x1, y1, x2, y2]
            f_width = f_box[2] - f_box[0]
            f_height = f_box[3] - f_box[1]
            
            f_x_center = f_box[0] + (f_width / 2)
            f_y_center = f_box[1] + (f_height / 2) # Get Y center too!
            face_area = f_width * f_height
            
            top_emotion = face.get('top_emotion')
            probability = face['probabilities'].get(top_emotion) if top_emotion and 'probabilities' in face else None
            landmarks = face.get('landmarks')
            
            # 2. MATCH THE BODY TO THE FACE
            matched_body_bbox = None
            matched_pose_keypoints = None
            
            if isinstance(poses, list):
                for person_body in poses:
                    if isinstance(person_body, dict)and len(person_body) > 0:
                        b_box = person_body['bbox'] 
                        
                        # is the center of the face inside this body's bounding box?
                        #note that b_box[3] > bbox[1] because y axis goes down
                        if (b_box[0] <= f_x_center <= b_box[2]) and (b_box[1] <= f_y_center <= b_box[3]):
                            matched_body_bbox = b_box
                            matched_pose_keypoints = person_body.get('pose') # The 17x3 matrix
                            break 
            
            # 3. Append everything safely
            master_data.append({
                'frame': frame_id,
                'frame_number': frame_number,
                'people_count': people_count,
                'person_index': i,
                'face_X_center': f_x_center,
                'face_area': face_area,
                'face_bbox': f_box,
                'body_bbox': matched_body_bbox,
                'top_emotion': top_emotion,
                'landmarks': landmarks,
                'pose_keypoints': matched_pose_keypoints,
                **{f'prob_{e}': p for e, p in face['probabilities'].items()}})
            
    else:
        # Handle empty frames
        master_data.append({
            'frame': frame_id,
            'frame_number': frame_number,
            'people_count': 0,
            'person_index': None,
            'face_X_center': None,
            'face_area': None,
            'face_bbox': None,
            'body_bbox': None,
            'top_emotion': None,
            'landmarks': None,
            'pose_keypoints': None
        })
df_master = pd.DataFrame(master_data)
print(f"Important data (df shape): {df_master.shape}")
print(df_master['pose_keypoints'].isna().sum())
print(f"Total rows: {len(df_master)}")

In [ ]:
KEYPOINTS = {
    'nose': 0, 'left_eye': 1, 'right_eye': 2,
    'left_ear': 3, 'right_ear': 4,
    'left_shoulder': 5, 'right_shoulder': 6,
    'left_elbow': 7, 'right_elbow': 8,
    'left_wrist': 9, 'right_wrist': 10,
    'left_hip': 11, 'right_hip': 12
}

def extract_pose_features(keypoints):
    if keypoints is None:
        return pd.Series({
            'shoulder_slope': None,
            'body_openness': None,
            'head_tilt': None,
            'wrist_height': None,
            'torso_height': None
        })
    
    kp = np.array(keypoints)  # 17x3: [x, y, visibility]
    
    l_shoulder = kp[5]
    r_shoulder = kp[6]
    l_wrist    = kp[9]
    r_wrist    = kp[10]
    l_hip      = kp[11]
    r_hip      = kp[12]
    nose       = kp[0]
    
    # slope between shoulders (positive = left higher, negative = right higher)
    shoulder_slope = r_shoulder[1] - l_shoulder[1]
    
    # horizontal distance between shoulders (wider = more open posture)
    body_openness = abs(r_shoulder[0] - l_shoulder[0])
    
    # nose x relative to shoulder midpoint (head tilt left/right)
    shoulder_mid_x = (l_shoulder[0] + r_shoulder[0]) / 2
    head_tilt = nose[0] - shoulder_mid_x
    
    # average wrist y relative to shoulder y (negative = wrists raised)
    shoulder_mid_y = (l_shoulder[1] + r_shoulder[1]) / 2
    wrist_height = shoulder_mid_y - ((l_wrist[1] + r_wrist[1]) / 2)
    
    # torso height: distance between shoulder midpoint and hip midpoint
    hip_mid_y = (l_hip[1] + r_hip[1]) / 2
    torso_height = abs(hip_mid_y - shoulder_mid_y)
    
    return pd.Series({
        'shoulder_slope': shoulder_slope,
        'body_openness': body_openness,
        'head_tilt': head_tilt,
        'wrist_height': wrist_height,
        'torso_height': torso_height
    })

# Apply to df_master, skipping rows with no pose
df_master_valid = df_master[df_master['pose_keypoints'].notna()].copy()
pose_features = df_master_valid['pose_keypoints'].apply(extract_pose_features)
df_master_valid = pd.concat([df_master_valid, pose_features], axis=1)

print(f"Valid rows for pose-emotion correlation: {len(df_master_valid)}")
print(df_master_valid[['shoulder_slope', 'body_openness', 'head_tilt', 'wrist_height', 'torso_height']].describe().round(2))

## Assemble the master dataset
Per-face visual + identity (`final_name`) + per-frame audio (nullable) + flags. `speaking_face` = this face is the person the audio says is talking.

In [ ]:
emotion_cols      = [c for c in df_master_valid.columns if c.startswith('prob_')]
pose_feature_cols = ['shoulder_slope','body_openness','head_tilt','wrist_height','torso_height']

labels = (pd.read_pickle('labels.pkl')
            .rename(columns={'Frame_Number':'frame_number', 'Person_Index':'person_index'}))
labels = labels.dropna(subset=['person_index']).copy()
labels['frame_number'] = labels['frame_number'].astype(int)
labels['person_index'] = labels['person_index'].astype(int)
df_master_valid['frame_number'] = df_master_valid['frame_number'].astype(int)
df_master_valid['person_index'] = df_master_valid['person_index'].astype(int)

df_all = df_master_valid.merge(
    labels[['frame_number','person_index','final_name','speaker']],
    on=['frame_number','person_index'], how='left')
df_all['speaker'] = df_all['speaker'].replace({'host':'moderator'})

df_all = df_all.merge(audio_by_frame, on='frame_number', how='left')
df_all['has_audio']   = df_all['has_audio'].fillna(False)
df_all['segment_id']  = df_all['segment_id'].fillna(-1).astype(int)
df_all['speaking_face'] = df_all['has_audio'] & (df_all['final_name'] == df_all['speaker'])

print('SHAPE:', df_all.shape, '| identity:', int(df_all.final_name.notna().sum()),
      '| has_audio:', int(df_all.has_audio.sum()), '| speaking:', int(df_all.speaking_face.sum()))

In [ ]:
# one row per audio frame (a frame ~ a second)
frames = df_all[df_all['has_audio']].drop_duplicates('frame_number')
print('SPEAKING TIME (audio-seconds) per speaker:')
print(frames['speaker'].value_counts(), '\n')

# total diarization segments per speaker (camera-independent)
seg_owner = df_all[df_all['has_audio']].groupby('segment_id')['speaker'].first()
print('TOTAL segments per speaker:')
print(seg_owner.value_counts(), '\n')

# segments where the speaker was actually visible (what seg can use)
print('ON-CAMERA-while-speaking segments (what seg keeps):')
print(seg['final_name'].value_counts())

## Segment-level view (for all audio↔visual analysis)
One row per segment: speaking-face emotion/pose **averaged** over the segment, the segment's audio value (`first`, since it's constant), and the segment's dominant emotion.

The audio features are one scalar per segment, but emotion and pose vary frame by frame. To correlate them honestly we need them at the same unit, and that unit has to be the segment (otherwise we'd duplicate each audio value across all its frames). So for every segment we collapse the visual side down to a single summary — the speaking face's average emotion and pose over that segment — and pair it with the segment's one audio value.

In [ ]:
def mode_or_nan(s):
    '''Returns the most common emotion in the segment, if the group is empty it returns NaN'''
    m = s.mode()
    return m.iloc[0] if len(m) else np.nan

spk = df_all[df_all['speaking_face']].copy() # spk = only the frames where we have a visible speaker with clean audio, so we can correlate their pose and emotion with what they are saying and how they are labeled

agg = {c: 'mean' for c in emotion_cols + pose_feature_cols}
agg.update({c: 'first' for c in NON_REDUNDANT})
agg['final_name']   = 'first'
agg['top_emotion']  = mode_or_nan
agg['frame_number'] = 'count'

seg = (spk.groupby('segment_id').agg(agg)
          .rename(columns={'frame_number': 'n_frames'})
          .reset_index())

print('segments with a visible speaker:', len(seg))
print('median frames per segment      :', int(seg['n_frames'].median()))
print(seg['final_name'].value_counts())

> **▲ END — manual pipeline.** All cells between **▼ START** and here are replaced by the pipeline alternative above.

The variables `seg`, `emotion_cols`, and `pose_feature_cols` are now available regardless of which approach you used. The analysis sections below use only these three — `df_all` is only needed by Section A.

In [ ]:
emotion_colors = {'Anger':'tab:red','Happiness':'tab:olive','Neutral':'tab:gray',
                  'Sadness':'tab:blue','Surprise':'tab:orange','Fear':'tab:purple',
                  'Disgust':'tab:green','Contempt':'tab:brown'}
emotion_order  = ['Anger','Disgust','Contempt','Fear','Sadness','Surprise','Happiness','Neutral']

def heat(cross, title, xlabel='Emotion', ylabel='Audio Feature', figsize=(12,6)):
    plt.figure(figsize=figsize)
    sns.heatmap(cross.astype(float), annot=True, fmt='.2f', cmap='coolwarm',
                vmin=-1, vmax=1, linewidths=0.5)
    plt.title(title); plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.tight_layout(); plt.show()

def within_speaker_corr(df, rows, cols, group='final_name', method='spearman'):
    g = df.dropna(subset=[group])
    out = pd.DataFrame(index=rows, columns=cols, dtype=float)
    for r in rows:
        for c in cols:
            d = g[[r, c, group]].dropna().copy()
            d['rx'] = d[r] - d.groupby(group)[r].transform('mean')
            d['ry'] = d[c] - d.groupby(group)[c].transform('mean')
            out.loc[r, c] = d['rx'].corr(d['ry'], method=method)
    return out

def eta_squared(df, value, factor):
    d = df[[value, factor]].dropna()
    if d[factor].nunique() < 2: return np.nan
    grand = d[value].mean()
    ssb = sum(len(g)*(g[value].mean()-grand)**2 for _, g in d.groupby(factor))
    sst = ((d[value]-grand)**2).sum()
    return ssb/sst if sst > 0 else np.nan

from scipy.stats import spearmanr
def corr_pvals(df, rows, cols):
    cc = pd.DataFrame(index=rows, columns=cols, dtype=float)
    pv = pd.DataFrame(index=rows, columns=cols, dtype=float)
    for r in rows:
        for c in cols:
            d = df[[r, c]].dropna()
            rho, p = spearmanr(d[r], d[c])
            cc.loc[r, c] = rho; pv.loc[r, c] = p
    return cc, pv

def heat_sig(cc, pv, title, xlabel='', ylabel='', alpha=0.05, figsize=(14,6)):
    ann = pd.DataFrame(index=cc.index, columns=cc.columns)
    for r in cc.index:
        for c in cc.columns:
            ann.loc[r, c] = f'{cc.loc[r,c]:.2f}\n*' if pv.loc[r,c] <= alpha else f'{cc.loc[r,c]:.2f}\n(ns)'
    plt.figure(figsize=figsize)
    sns.heatmap(cc.astype(float), annot=ann, fmt='', cmap='coolwarm', vmin=-1, vmax=1,
                linewidths=0.5, annot_kws={'size':8})
    plt.title(f'{title}\n(* = p<{alpha}, ns = not significant)')
    plt.xlabel(xlabel); plt.ylabel(ylabel); plt.tight_layout(); plt.show()

# faces with a real identity (exclude 'uncertain') for visual-only work
ident = df_all[df_all['final_name'].notna() & (df_all['final_name'] != 'uncertain')].copy()
candidates = [n for n in ['Martins','Gouveia_Melo','moderator'] if n in ident['final_name'].unique()]
print('candidates:', candidates)

## A. Visual ↔ Visual  (frame level — one row per face)

### A1. Pose ↔ Emotion

In [ ]:
cc = ident[pose_feature_cols + emotion_cols].corr(method='spearman').loc[pose_feature_cols, emotion_cols]
heat(cc, 'Pose vs Emotion (frame level, all identified faces)', ylabel='Pose Feature')

### A2. Emotion distribution per candidate

In [ ]:
for name in candidates:
    sub = ident[ident['final_name'] == name]
    plt.figure(figsize=(8,4))
    sns.countplot(data=sub, x='top_emotion', order=emotion_colors.keys(),
                  palette=emotion_colors, hue='top_emotion', legend=False)
    plt.title(f'{name} — top-emotion distribution ({len(sub)} faces)')
    plt.xlabel(''); plt.ylabel('count'); plt.xticks(rotation=45)
    plt.tight_layout(); plt.show()

### A3. Emotion over time, per candidate (smoothed)

In [ ]:
WINDOW = 30
prob_cols = [f'prob_{e}' for e in emotion_colors]
fig, axes = plt.subplots(len(candidates), 1, figsize=(16, 3*len(candidates)), sharex=True)
axes = np.atleast_1d(axes)
for ax, name in zip(axes, candidates):
    sub = ident[ident['final_name'] == name]
    sm = sub.groupby('frame_number')[prob_cols].mean().rolling(WINDOW, center=True, min_periods=1).mean()
    for e in emotion_colors:
        ax.plot(sm.index, sm[f'prob_{e}'], color=emotion_colors[e], label=e, linewidth=1.3, alpha=0.85)
    ax.set_title(f'{name}'); ax.set_ylabel('prob'); ax.grid(alpha=0.3)
axes[-1].set_xlabel('frame number (seconds)')
axes[0].legend(fontsize=8, bbox_to_anchor=(1.01,1), loc='upper left')
plt.suptitle('Smoothed emotion probabilities over time'); plt.tight_layout(); plt.show()

## B. Audio ↔ Visual  (segment level — one row per segment)

### B1. Audio ↔ Emotion (all candidates)

In [ ]:
cc = seg[NON_REDUNDANT + emotion_cols].corr(method='spearman').loc[NON_REDUNDANT, emotion_cols]
heat(cc, f'Audio vs Emotion (segment level, n={len(seg)} segments)')

### B2. Audio ↔ Emotion, per candidate

In [ ]:
for name in candidates:
    s = seg[seg['final_name'] == name]
    if len(s) < 10: 
        print(f'{name}: only {len(s)} segments, skipping'); continue
    cc = s[NON_REDUNDANT + emotion_cols].corr(method='spearman').loc[NON_REDUNDANT, emotion_cols]
    heat(cc, f'Audio vs Emotion — {name} ({len(s)} segments)')

### B3. Audio feature distributions by dominant emotion

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9)); axes = axes.flatten()
for i, feat in enumerate(NON_REDUNDANT):
    sns.boxplot(data=seg, x='top_emotion', y=feat, order=emotion_order,
                palette=emotion_colors, hue='top_emotion', legend=False,
                ax=axes[i], linewidth=0.8, fliersize=2)
    axes[i].set_title(feat); axes[i].set_xlabel(''); axes[i].set_ylabel('')
    axes[i].tick_params(axis='x', rotation=45, labelsize=8); axes[i].grid(axis='y', alpha=0.3)
fig.suptitle('Audio feature distributions by dominant emotion (segment level)', fontsize=14)
plt.tight_layout(); plt.show()

### B4. Mean audio per dominant emotion (z-scored)

In [ ]:
from scipy.stats import zscore
summary = seg.groupby('top_emotion')[NON_REDUNDANT].mean().round(2)
print(summary)
summary_z = seg.groupby('top_emotion')[NON_REDUNDANT].mean().apply(zscore)
plt.figure(figsize=(12,6))
sns.heatmap(summary_z, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Mean audio features per dominant emotion (z-scored, segment level)')
plt.tight_layout(); plt.show()

### B5. Audio ↔ Pose (speaking face only)

In [ ]:
cc = seg[pose_feature_cols + NON_REDUNDANT].corr(method='spearman').loc[pose_feature_cols, NON_REDUNDANT]
heat(cc, 'Pose vs Audio (segment level, speaking face)', xlabel='Audio Feature', ylabel='Pose Feature', figsize=(14,5))

## C. Is the signal real, or just *who* is speaking?

### C1. Within-speaker correlation (identity controlled)
Audio↔emotion after removing each candidate's mean. If this collapses toward 0 vs B1, the pooled correlations were just differences **between** candidates.

In [ ]:
wsc = within_speaker_corr(seg, NON_REDUNDANT, emotion_cols)
heat(wsc, 'Audio vs Emotion — WITHIN speaker (segment level, identity controlled)')

### C2. Variance partition (η²): explained by *who* vs *which emotion*

In [ ]:
var_part = pd.DataFrame({
    'by_speaker': {f: eta_squared(seg, f, 'final_name')  for f in NON_REDUNDANT},
    'by_emotion': {f: eta_squared(seg, f, 'top_emotion') for f in NON_REDUNDANT},
}).round(3)
print(var_part)

### C3. Significance (segment level, valid now that rows are independent)

In [ ]:
cc, pv = corr_pvals(seg, NON_REDUNDANT, emotion_cols)
heat_sig(cc, pv, 'Audio vs Emotion (segment level)', xlabel='Emotion', ylabel='Audio Feature')

cc, pv = corr_pvals(ident, pose_feature_cols, emotion_cols)
heat_sig(cc, pv, 'Pose vs Emotion (frame level)', xlabel='Emotion', ylabel='Pose Feature')

cc, pv = corr_pvals(seg, pose_feature_cols, NON_REDUNDANT)
heat_sig(cc, pv, 'Pose vs Audio (segment level)', xlabel='Audio Feature', ylabel='Pose Feature')

## D. Is one modality enough?

In [ ]:
from sklearn.cross_decomposition import CCA
from scipy.stats import pearsonr

visual_features = pose_feature_cols + emotion_cols
audio_features  = NON_REDUNDANT
d = seg[visual_features + audio_features].dropna()
print('segments used:', len(d))

# structure of each modality space
mods = {'Visual (pose+emotion)': visual_features, 'Audio': audio_features,
        'All combined': visual_features + audio_features}
fig, axes = plt.subplots(1, 3, figsize=(21,6))
for ax, (nm, fs) in zip(axes, mods.items()):
    X2 = PCA(2, random_state=42).fit_transform(StandardScaler().fit_transform(d[fs]))
    ax.scatter(X2[:,0], X2[:,1], c='steelblue', alpha=0.4, s=15, edgecolors='none')
    ax.set_title(nm); ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.grid(alpha=0.3)
plt.suptitle('PCA structure per modality (segments)'); plt.tight_layout(); plt.show()

# how much do the modalities actually SHARE?
Xv = StandardScaler().fit_transform(d[visual_features])
Xa = StandardScaler().fit_transform(d[audio_features])
ncomp = min(3, Xv.shape[1], Xa.shape[1])
Uv, Ua = CCA(n_components=ncomp).fit(Xv, Xa).transform(Xv, Xa)
print('\nCanonical correlations (visual vs audio):')
for i in range(ncomp):
    r, p = pearsonr(Uv[:,i], Ua[:,i])
    print(f'  component {i+1}: r = {r:.3f}  (p = {p:.4f})')
print('\nLow values -> modalities carry different information -> one modality is NOT enough.')